# Stage 4: persistent ChromaDB index

Persistent indexing avoids re-embedding the corpus at every startup. The collection stores stable corpus IDs, exact retrieval text, caller-provided embeddings, and a scalar metadata projection. It contains no evaluation queries.

In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
selected = json.loads((ROOT / 'data/processed/selected_embedding_model.json').read_text(encoding='utf-8'))
manifest = json.loads((ROOT / 'data/processed/chroma_index_manifest.json').read_text(encoding='utf-8'))
print('Selected configuration:', selected['config_id'])
print('Collection:', manifest['collection_name'])
print('Expected records:', manifest['corpus_record_count'])

## Scalar metadata projection

The source JSONL remains the rich source of truth. Chroma receives only complete scalar fields, including level-support booleans and a deterministic JSON string for source IDs.

In [ ]:
from src.dataset_io import read_jsonl
from src.metadata_projection import project_metadata

records = read_jsonl(ROOT / 'data/processed/knowledge_corpus.jsonl')
print(project_metadata(records[0]))

## Reopen and verify persistence

A fresh `PersistentClient` is opened here. The reusable verifier checks every ID, document, projected metadata object, count, and manifest marker.

In [ ]:
from src.chroma_store import reopen_and_verify

persistence = reopen_and_verify(project_root=ROOT, chroma_path=ROOT / 'chroma_db', records=records, manifest=manifest)
print(persistence)

## Semantic and filtered queries

The selected model creates query embeddings explicitly; Chroma's default embedding function is never used. The unfiltered examples do not use oracle metadata. Separate examples demonstrate documented filters.

In [ ]:
import chromadb
from src.embedding_models import load_embedding_configurations, load_sentence_transformer, encode_texts

settings, configurations = load_embedding_configurations(ROOT / 'config/embedding_models.json')
configuration = next(item for item in configurations if item.config_id == selected['config_id'])
model = load_sentence_transformer(configuration.model_id, selected['device'], revision=selected['model_revision'], local_files_only=True)
queries = ['How can I debug a program that crashes?', 'How should I study mathematics problems?']
vectors = encode_texts(model, [configuration.format_query(text) for text in queries], batch_size=2, normalize_embeddings=True)
client = chromadb.PersistentClient(path=str(ROOT / 'chroma_db'))
collection = client.get_collection(manifest['collection_name'], embedding_function=None)
plain = collection.query(query_embeddings=vectors.tolist(), n_results=5, include=['documents', 'metadatas', 'distances'])
for text, ids in zip(queries, plain['ids']):
    print(text, ids)
filtered = collection.query(query_embeddings=[vectors[0].tolist()], where={'subject_family': 'computing'}, n_results=5, include=['metadatas'])
print('Computing filter:', filtered['ids'][0])
assert all(item['subject_family'] == 'computing' for item in filtered['metadatas'][0])
client.close()

## Direct cosine comparison and conclusion

The integration verifier compares unfiltered Chroma top tens with direct normalized cosine top tens on the 64 development queries. The result below summarizes logical persistence, overlap, and all filter checks. The final-test set is still untouched.

In [ ]:
summary = json.loads((ROOT / 'data/processed/embedding_benchmark_summary.json').read_text(encoding='utf-8'))
verification = summary['chroma_verification']
print('Average top-10 overlap:', verification['average_direct_chroma_top_10_overlap'])
print('Persistence:', verification['persistence'])
print('Filters:', {name: item['passed'] for name, item in verification['metadata_filters'].items()})
print('Final-test metrics computed:', verification['sealed_final_test_metrics_computed'])